# CIRCUIT 2.0 — Fine-Tuning Pipeline (Meta Llama-3.1-8B-Instruct via QLoRA)
This notebook fine-tunes Meta-Llama-3.1-8B-Instruct on the CIRCUIT 2.0 criminal intelligence and forensic analysis dataset.

### Capabilities Trained:
1. **Bilingual FIR Extraction & NER** (IPC / BNS statutes, suspects, vehicles, bank accounts)
2. **Smurfing & Structuring PAN Evasion Detection** (PMLA Sec 3 & Income Tax Act Sec 139A)
3. **Call Detail Records (CDR) Nocturnal Surveillance** (Tower handshakes, unusual off-hours calls)
4. **Syndicate Bottleneck Disruption Analysis** (Network articulation points & PageRank)
5. **Statutory Electronic Evidence Certification** (BSA Section 63 / Section 65B IEA)

In [ ]:
# 1. Install Deep Learning & QLoRA Dependencies
!pip install -q torch transformers peft trl bitsandbytes datasets accelerate

In [ ]:
# 2. Clone CIRCUIT 2.0 Repository & Prepare Data
!git clone https://github.com/ei-sanu/circuit2.0.git
%cd circuit2.0
!python scripts/export_finetune_dataset.py

In [ ]:
# 3. Execute QLoRA Fine-Tuning Script
!python scripts/train_fine_tune.py \
    --base-model meta-llama/Meta-Llama-3.1-8B-Instruct \
    --train-data data/train.jsonl \
    --val-data data/val.jsonl \
    --output-dir checkpoints/circuit_llama3_lora \
    --epochs 3 \
    --batch-size 2 \
    --grad-accum 4 \
    --lr 2e-4 \
    --lora-r 16 \
    --lora-alpha 32

In [ ]:
# 4. Test Fine-Tuned Inference in Python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model = "meta-llama/Meta-Llama-3.1-8B-Instruct"
adapter_path = "checkpoints/circuit_llama3_lora"

tokenizer = AutoTokenizer.from_pretrained(base_model)
model = AutoModelForCausalLM.from_pretrained(base_model, torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(model, adapter_path)

prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are CIRCUIT AI, an expert criminal intelligence copilot.<|eot_id|><|start_header_id|>user<|end_header_id|>
Analyze the financial transaction chain: Account ACC-918237 received ₹48,500 and ₹49,000 within 4 hours; immediately forwarded ₹95,000 to Apex Factoring LLC.<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.2)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
